In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l1.4 Cross-entropy

Cross-entropy is the training loss: the negative log-probability the model
assigned to the correct next token. Lower means the model was more right.

In [ ]:
from torch.nn import functional as F

logits = torch.tensor([[2.0, 0.5, 0.1]])
target = torch.tensor([0])
by_hand = -F.log_softmax(logits, dim=-1)[0, target.item()]
builtin = F.cross_entropy(logits, target)
print("by hand:", by_hand.item(), " F.cross_entropy:", builtin.item())

A useful sanity anchor: a model guessing uniformly over a vocab of size V
scores `ln(V)`. Beating that baseline is the first sign of learning.

In [ ]:
assert torch.allclose(by_hand, builtin)